# Lecture 2: Multiple Linear Regression - California Housing

### Short, cell-by-cell, self-study notes

Multiple Linear Regression predicts one numeric target using two or more input features. Here, we predict median California house value from eight housing features.

**Main idea:** The model learns one intercept and one coefficient for every input feature.

## 1. Simple versus multiple linear regression

| Type | Input features | Example |
|---|---|---|
| Simple Linear Regression | One feature | Predict height from weight |
| Multiple Linear Regression | Two or more features | Predict house value from income, rooms, location, and more |

The multiple-regression formula is:

**predicted price = intercept + c1x1 + c2x2 + ... + c8x8**

Each coefficient shows the direction and size of a feature effect while the other features are held fixed.

## 2. Cell 1: import tools and load the dataset

The supplied practice notebook creates a dataframe step by step from raw arrays. This clearer version uses as_frame=True, which gives a ready pandas dataframe.

The target is named MedHouseVal in the source dataset. We rename it to Price so its purpose is obvious.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "Price"})

print("Dataset shape:", df.shape)
print("Target:", housing.target_names[0])
df.head()

### What changed from the supplied notebook?

- The original notebook uses California data arrays and then manually builds a dataframe.
- This version requests a dataframe directly, so column names and target stay together.
- The rest of the machine-learning workflow is the same: inspect, split, scale, train, evaluate, and save.

## 3. Cell 2: quick data checks

Before modelling, check data types, missing values, and basic ranges. Every feature in this dataset is numeric, and Price is the value to predict.

In [ ]:
quality_check = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "minimum": df.min(),
    "maximum": df.max(),
})

quality_check

## 4. Cell 3: correlation heatmap

A correlation heatmap gives a quick view of linear relationships.

- Positive values mean two features tend to rise together.
- Negative values mean one tends to fall when the other rises.
- The Price row or column shows which features have stronger linear relationships with the target.

Correlation is a clue, not proof of cause. It also does not describe non-linear relationships.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("California Housing feature correlation heatmap", weight="bold")
plt.show()

df.corr()["Price"].sort_values(ascending=False).round(3)

## 5. Cell 4: separate X and y, then split

- X contains all eight input features.
- y is Price.
- One third of the data is kept for testing in this example.

Use the same split for every model comparison. Otherwise a score difference might be caused by different test rows instead of a different model.

In [ ]:
X = df.drop(columns="Price")
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=10
)

print("Training shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

## 6. Cell 5: train a safe pipeline

The supplied notebook scales training data and test data in separate lines. This version uses a pipeline:

**input data → StandardScaler → LinearRegression → prediction**

A pipeline keeps the steps in the correct order. It learns scaling from X_train during fit, then uses the same learned scaling for X_test during predict. This prevents data leakage.

In [ ]:
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

regressor = model.named_steps["linearregression"]
coefficient_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient_after_scaling": regressor.coef_,
}).sort_values("coefficient_after_scaling", key=np.abs, ascending=False)

print("Intercept:", round(regressor.intercept_, 3))
coefficient_table

### How to read coefficients

Because features were standardized, coefficient sizes are easier to compare.

- Positive coefficient: increasing the feature tends to increase predicted Price.
- Negative coefficient: increasing the feature tends to decrease predicted Price.
- A coefficient is not proof of cause. Correlated features can share or distort apparent effects.

## 7. Cell 6: evaluate the test predictions

We evaluate only on test rows that the model did not see during fitting.

- MAE is an average absolute price error.
- RMSE is a larger-error-sensitive price error.
- R² shows the fraction of variation explained by the model.
- Adjusted R² accounts for the number of input features.

In [ ]:
mse = mean_squared_error(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
test_r2 = r2_score(y_test, y_pred_test)

n_test = len(y_test)
number_of_features = X_test.shape[1]
adjusted_r2 = 1 - (1 - test_r2) * (n_test - 1) / (n_test - number_of_features - 1)

metrics = pd.Series({
    "train_R2": r2_score(y_train, y_pred_train),
    "test_R2": test_r2,
    "adjusted_test_R2": adjusted_r2,
    "test_MAE": mae,
    "test_RMSE": rmse,
}).round(3)
metrics

## 8. Cell 7: prediction and residual diagrams

In multiple regression we cannot draw one single best-fit line because there are eight input dimensions. Instead, we use diagnostic diagrams:

- Actual versus predicted: dots closer to the diagonal are better.
- Residual plot: errors should be scattered around zero without a strong pattern.

A visible pattern means the straight-line model may be missing an important relationship.

In [ ]:
residuals = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(y_test, y_pred_test, alpha=0.35, color="#457b9d")
low = min(y_test.min(), y_pred_test.min())
high = max(y_test.max(), y_pred_test.max())
axes[0].plot([low, high], [low, high], "--", color="#e76f51")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
axes[0].set_title("Actual versus predicted")

axes[1].scatter(y_pred_test, residuals, alpha=0.35, color="#457b9d")
axes[1].axhline(0, linestyle="--", color="#e76f51")
axes[1].set_xlabel("Predicted Price")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals around zero are preferred")

fig.suptitle("Multiple Linear Regression diagnostic diagrams", weight="bold")
fig.tight_layout()
plt.show()

## 9. Cell 8: save the full pipeline

Pickling serializes an object so it can be stored and later loaded. Save the whole pipeline, not only the regression model. Then a deployed prediction automatically applies the same scaler before predicting.

The following cell creates a local file named california_house_price_pipeline.pkl. Run it only when you want to save the trained model.

In [ ]:
import pickle

model_path = "california_house_price_pipeline.pkl"
with open(model_path, "wb") as file:
    pickle.dump(model, file)

with open(model_path, "rb") as file:
    loaded_model = pickle.load(file)

print("Saved and loaded pipeline gives the same first prediction:")
print(round(loaded_model.predict(X_test.iloc[[0]])[0], 3))

## 10. Final revision card

- Multiple Linear Regression uses two or more independent features.
- California Housing has eight numeric input features and Price as the target.
- Inspect missing values and relationships before modelling.
- Split into training and test data before fitting preprocessing or a model.
- A pipeline safely applies scaling and regression in the correct order.
- There is one coefficient per input feature and one intercept.
- Use MAE, RMSE, R², adjusted R², and residual plots to evaluate the model.
- Save the full pipeline for deployment so preprocessing is not forgotten.

### One-line interview answer

**Multiple Linear Regression predicts one numeric target using several features, with one learned coefficient per feature, and we evaluate it on unseen test data using error metrics and residual diagnostics.**